In [2]:
# Cell 1 — setup
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
import os
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=ValueWarning)

from scripts.data_loader import MacroDataLoader
from scripts.validator import WalkForwardValidator
from scripts.evaluator import ModelEvaluator



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Cell 2 — load data
loader = MacroDataLoader('../data/master_data1.csv')
loader.load()
loader.build_lag_features()
X, y = loader.get_feature_matrix()
print(X.shape, y.shape)

(160, 6) (160,)


In [4]:
# Training data before 2022=01-01 and testing on the data beyond that
X_train = X[X.index < '2022-01-01']
X_test = X[X.index >= '2022-01-01']
y_train = y[y.index < '2022-01-01']
y_test = y[y.index >= '2022-01-01']

print(X_train.shape, X_test.shape)

(108, 6) (52, 6)


In [5]:
tscv = TimeSeriesSplit(n_splits=5)

#params = {'n_estimators': [100, 200, 300], 'max_depth': [3, 4, 5], 'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.8, 1.0]}
params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3],
    'learning_rate': [0.01, 0.05],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.5, 0.7],
    'min_child_weight': [3, 5] }

xgb = XGBRegressor(random_state=42)

grid_search = GridSearchCV(xgb, params, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV RMSE:", round(-grid_search.best_score_, 4))

Best params: {'colsample_bytree': 0.5, 'learning_rate': 0.05, 'max_depth': 2, 'min_child_weight': 5, 'n_estimators': 100, 'subsample': 0.6}
Best CV RMSE: 1.4949


In [6]:
# Walk forward validation with best params
model = XGBRegressor(**grid_search.best_params_, random_state=42)
validator = WalkForwardValidator(model, X, y, test_start='2022-01-01')
validator.run()
print(validator.get_metrics())

{'rmse': np.float64(0.839), 'mae': 0.6607}


In [7]:
# Overfitting check
from sklearn.metrics import mean_squared_error, mean_absolute_error

in_sample_pred = model.predict(X_train)
in_sample_rmse = np.sqrt(mean_squared_error(y_train, in_sample_pred))
print(f"In sample RMSE:  {round(in_sample_rmse, 4)}")
print(f"Out sample RMSE: {round(validator.get_metrics()['rmse'], 4)}")
print(f"MAE:             {round(validator.get_metrics()['mae'], 4)}")

In sample RMSE:  0.5901
Out sample RMSE: 0.839
MAE:             0.6607


## Post-2016 Robustness Check — Finding
Restricting training to post-2016 FIT regime worsens forecast 
accuracy — RMSE 0.8828 versus 0.8390 for full sample.
Pre-2016 volatile regime data improves model generalization 
by teaching XGBoost to recognise deviations from anchored inflation.
Full sample training is the preferred specification.

## ARIMA-XGBoost Hybrid Model
ARIMA captures linear autoregressive structure.
XGBoost models ARIMA residuals using macro features.
Final forecast = ARIMA prediction + XGBoost residual correction.

In [8]:
from statsmodels.tsa.arima.model import ARIMA

# Hybrid model — ARIMA captures linear structure, XGBoost corrects residuals
ext_cols = ['crude_lag1', 'inr_lag1', 'm3_lag12']

# Step 1 — ARIMA residuals on training data
arima_model = ARIMA(y_train, order=(3,0,0)).fit()
residuals = arima_model.resid

# Step 2 — XGBoost trains on residuals using external features only
xgb_resid = XGBRegressor(**grid_search.best_params_, random_state=42)
xgb_resid.fit(X_train[ext_cols], residuals)

# Step 3 — walk forward hybrid
train_y = y_train.copy()
predictions_hybrid = []
X_test = X[X.index >= '2022-01-01']
y_test = y[y.index >= '2022-01-01']

for t in range(len(X_test)):
    arima_pred = ARIMA(train_y, order=(3,0,0)).fit().forecast(steps=1)[0]
    resid_pred = xgb_resid.predict(X_test[ext_cols].iloc[[t]])[0]
    predictions_hybrid.append(arima_pred + resid_pred)
    train_y = pd.concat([train_y, y_test.iloc[[t]]])

rmse_hybrid = np.sqrt(mean_squared_error(y_test, predictions_hybrid))
mae_hybrid = mean_absolute_error(y_test, predictions_hybrid)
print(f"Hybrid RMSE: {round(rmse_hybrid,4)} MAE: {round(mae_hybrid,4)}")

Hybrid RMSE: 0.7044 MAE: 0.5316


## Notebook 04 Summary — XGBoost and Hybrid Results



## Model Comparison
| Model                 | RMSE   | MAE    |
|----------------------|--------|--------|
| ARIMA(3,0,0)         | 0.7024 | 0.5397 |
| ARIMAX(3,0,0)        | 0.7246 | 0.5438 |
| XGBoost Full Sample  | 0.8390 | 0.6607 |
| ARIMA-XGBoost Hybrid | 0.7044 | 0.5316 |

## Key Findings

**XGBoost standalone underperforms ARIMA**
Despite careful feature engineering, SHAP validated feature 
selection, and extensive hyperparameter tuning, XGBoost could 
not beat AR(3) on RMSE. Indian CPI inflation is dominated by 
its own momentum, external macro variables add limited 
incremental forecasting power over pure autoregression on 
this sample size.

**Post-2016 restriction worsens performance**
Restricting training to FIT regime reduced accuracy RMSE 
0.8828 versus 0.8390 full sample. Pre-2016 volatile regime 
teaches the model to recognise deviations from anchored 
inflation. Full sample is the preferred specification.

**ARIMA-XGBoost Hybrid is the best model**
Hybrid achieves lowest MAE of 0.5316 beating ARIMA 0.5390.
ARIMA captures linear autoregressive structure perfectly.
XGBoost corrects residual errors using crude oil, exchange 
rate, and M3, three external channels ARIMA is blind to.
Together they produce more consistent forecasts than either alone.

## Why XGBoost Did Not Beat ARIMA

Three honest reasons:
1. Sample size :— 108 training observations is small for tree 
   based models to reliably extract non-linear signal
2. AR(3) is an exceptionally strong baseline, inflation 
   persistence dominates all external signals at short horizons
3. External variable relationships are time varying ,
   sometimes crude oil dominates, sometimes M3, sometimes 
   neither. XGBoost struggles to learn switching behavior 
   on limited data

## SHAP Contribution
Despite RMSE limitation SHAP revealed economically meaningful 
time varying feature importance, crude oil spiking precisely 
at Russia-Ukraine 2022, M3 lag 12 confirming P1 monetary 
transmission finding, and CPI lag 1 regime switching behavior. 
These findings are the paper's core economic contribution 
independent of forecast accuracy comparison.